In [ ]:
#https://github.com/laxmimerit/LangGraph-and-Ollama/blob/main/02.%20Langchain%20Getting%20Started/Langchain%20Getting%20Started.ipynb

In [1]:
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
#SystemMessage->istruzioni globali(hanno piu autorita), HumanMessage->input dell'utente, AIMessage->output del modello
from langchain_ollama import ChatOllama  #importi wrapper langchain per ollama

In [ ]:
llm = ChatOllama(model="tinyllama:latest",  #"gemma3",
                  base_url="http://localhost:11434",
                  temperature=0,  #🔥TEMPERATURE! constrolla randomness/creatività del modello, 0->output piu deterministico, 1->output piu creativo, temperature influenza quanto il modello 'osa'
                  num_predict=1024,  #max numero token generabili in output (modello si ferma se supera quel numero)
                  num_ctx=2048)  #dimensione context window, lunghezza massima del contesto (input+output), se superi num_ctx il modello non riesce a processare input e output insieme, quindi devi ridurre input o num_predict

In [ ]:
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage(content="What is Langchain?"),
]

messages

In [ ]:

response = llm.invoke(messages)  #chiamata reale al modello (invoke esegue il Runnable).
#response è un un AIMessage(...)
response.pretty_print()  #metodo helper di langchain, stampa mex in modo leggibile.

In [ ]:

from langchain_core.prompts import ChatPromptTemplate  #ChatPromptTemplate x Template


prompt = ChatPromptTemplate([
    ("system", "You are expert in {subject}. Explain it to a {audience} students."),
    ("human", "Can you explain {topic}?")
]  #ogni tupla è (role, template)
)

messages = prompt.invoke({  #prompt è un Runnable, invoke esegue il Runnable, prende in input un dict con le variabili da sostituire nei template
    "subject": "Physics",
    "audience": "high school",
    "topic": "Quantum Mechanics"
})
messages

In [ ]:

response = llm.invoke(messages) 
response.pretty_print()


In [ ]:

messages = prompt.invoke({  #prompt è un Runnable, invoke esegue il Runnable, prende in input un dict con le variabili da sostituire nei template
    "subject": "Physics",
    "audience": "Ph. D.",
    "topic": "Quantum Mechanics"
})
response = llm.invoke(messages) 
response.pretty_print()

In [ ]:
# invoke -> langchain runnable
chain = prompt | llm   #runnable SEQUENCE, | è un operatore di concatenazione, output di a diventa input di b

response = chain.invoke({"subject": "Physics",   #usi ur 'chain'
                      "audience": "middle school",
                      "topic": "Relativity"})

response.pretty_print()

In [ ]:
from langchain_core.output_parsers import StrOutputParser #StrOUtputParser converte output del modello in una semplice stringa python (perche molti llm in langchain posso return un obj)

# invoke -> langchain runnable
chain = prompt | llm | StrOutputParser()  #runnable SEQUENCE, | è un operatore di concatenazione, output di a diventa input di b, aggiungo StrOutputParser alla fine della catena per convertire output del modello in una semplice stringa python

response = chain.invoke({"subject": "Physics",
                      "audience": "middle school",
                      "topic": "Relativity"})

print(response)  #ora visto che hai usato StrOutputParser, response è una semplice stringa python, quindi puoi stamparla direttamente senza pretty_print()

In [ ]:
#🔥LLM con output strutturato grazuie a pydantic

from pydantic import BaseModel, Field  #pydantic serve x definire strutture dati tipizzate, BaseModel è la classe base da cui ereditare, Field serve x definire campi con metadati (esempio descrizione)

class Sentiment(BaseModel):  #definisco una struttura dati tipizzata chiamata Sentiment, che eredita da BaseModel di pydantic
    sentiment:bool = Field(..., description="True if the sentiment is positive, False otherwise")  #campo booleano, ... indica che è un campo obbligatorio, description è una descrizione del campo (verra usata dal modello)
    reasoning:str = Field(..., description="A brief explanation of the sentiment")  #campo string


structured_llm = llm.with_structured_output(Sentiment)  #nuova istanza di llm che sa che deve restituire un output strutturato secondo la struttura dati Sentiment, quando chiami invoke su structured_llm, langchain sa che deve convertire output del modello in un oggetto Sentiment (usando le descrizioni dei campi per capire come mappare output del modello ai campi di Sentiment)

response = structured_llm.invoke("I love programming!")
response

response.model_dump()  #converte obj pydantic->dizionario python

#streaming
for chunk in structured_llm.stream("I love programming!"):
    print(chunk, end='', flush=True)
#end='' non va a capo ogni volta, flush=True forza stampa immediata

for chunk in llm.stream("I love programming! write a poem about it"):
    print(chunk.content, end='', flush=True)
#here usi il llm with structured output, usi il llm originale raw.    